In [1]:
# Cell 1 — Imports and setup
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('..')
import config

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

CITIES = list(config.CITY_SETTINGS.keys())
PARQUET_PATHS = {
    city: os.path.join(config.BASE_DIR, "data", city, f"{city}_silver_standard.parquet")
    for city in CITIES
}

print("Cities:", CITIES)
for city, path in PARQUET_PATHS.items():
    exists = "✅" if os.path.exists(path) else "❌ MISSING"
    size = f"{os.path.getsize(path)/1024:.1f} KB" if os.path.exists(path) else "N/A"
    print(f"  {exists}  {city}: {path} ({size})")

Cities: ['manhattan', 'pittsburgh', 'philadelphia']
  ✅  manhattan: /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data/manhattan/manhattan_silver_standard.parquet (843.5 KB)
  ✅  pittsburgh: /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data/pittsburgh/pittsburgh_silver_standard.parquet (128.6 KB)
  ✅  philadelphia: /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data/philadelphia/philadelphia_silver_standard.parquet (167.5 KB)


In [2]:
# Cell 2 — Load all parquet files
dfs = {}
load_errors = {}

for city, path in PARQUET_PATHS.items():
    try:
        dfs[city] = pd.read_parquet(path)
        print(f"✅ {city}: {len(dfs[city])} rows × {len(dfs[city].columns)} columns")
    except Exception as e:
        load_errors[city] = str(e)
        print(f"❌ {city}: FAILED — {e}")

if load_errors:
    print(f"\n⚠️  {len(load_errors)} files failed to load")

✅ manhattan: 7000 rows × 11 columns
✅ pittsburgh: 1023 rows × 11 columns
✅ philadelphia: 1278 rows × 11 columns


In [3]:
# Cell 3 — Row counts and label distributions
summary_rows = []

for city, df in dfs.items():
    row = {"city": city, "total_rows": len(df)}
    if 'oracle_label' in df.columns:
        counts = df['oracle_label'].value_counts()
        for label in ['Answerable', 'Contradictory', 'Ambiguous']:
            row[label] = counts.get(label, 0)
            row[f"{label}_%"] = f"{counts.get(label, 0)/len(df):.1%}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('city')
print("=== ROW COUNTS AND LABEL DISTRIBUTIONS ===")
display(summary_df)

=== ROW COUNTS AND LABEL DISTRIBUTIONS ===


,total_rows,Answerable,Answerable_%,Contradictory,Contradictory_%,Ambiguous,Ambiguous_%
city,,,,,,,
manhattan,7000,5468,78.1%,1532,21.9%,0,0.0%
pittsburgh,1023,705,68.9%,318,31.1%,0,0.0%
philadelphia,1278,990,77.5%,288,22.5%,0,0.0%


In [4]:
# Cell 4 — Schema overview per city
EXPECTED_COLUMNS = [
    'sample_id', 'city', 'instruction', 'oracle_label',
    'candidate_count', 'start_node', 'gold_goal_node',
    'extracted_category', 'extracted_noun', 'extracted_direction',
    'target_node',
    # enriched columns (may be missing if enrich_silver.py not run)
    'gold_goal_lat', 'gold_goal_lon',
]

for city, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"  {city.upper()} — Schema")
    print(f"{'='*60}")

    schema_rows = []
    for col in df.columns:
        null_count = df[col].isna().sum()
        null_pct = null_count / len(df)
        sample_vals = df[col].dropna().head(3).tolist()
        schema_rows.append({
            "column":    col,
            "dtype":     str(df[col].dtype),
            "nulls":     null_count,
            "null_%":    f"{null_pct:.1%}",
            "expected":  "✅" if col in EXPECTED_COLUMNS else "➕ extra",
            "sample":    str(sample_vals)[:80],
        })

    # Flag missing expected columns
    missing_expected = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing_expected:
        for col in missing_expected:
            schema_rows.append({
                "column":  col,
                "dtype":   "MISSING",
                "nulls":   "N/A",
                "null_%":  "N/A",
                "expected": "❌ MISSING",
                "sample":  "",
            })

    schema_df = pd.DataFrame(schema_rows).set_index('column')
    display(schema_df)


  MANHATTAN — Schema


,dtype,nulls,null_%,expected,sample
column,,,,,
sample_id,int64,0,0.0%,✅,"[316, 140, 457]"
city,object,0,0.0%,✅,"['manhattan', 'manhattan', 'manhattan']"
instruction,object,0,0.0%,✅,"[""Can you meet me at the garden on Liberty Street. It's ..."
oracle_label,object,0,0.0%,✅,"['Answerable', 'Answerable', 'Answerable']"
candidate_count,int64,0,0.0%,✅,"[122, 256, 659]"
start_node,object,0,0.0%,✅,"['#5515156304', '#2709347118', '#8958523249']"
gold_goal_node,object,0,0.0%,✅,"['#697619118', '#2711466517', '#3863997257']"
extracted_category,object,0,0.0%,✅,"['GARDEN', 'CAFE', 'RESTAURANT']"
extracted_noun,object,1417,20.2%,✅,"['garden', 'cafe', 'restaurant']"



  PITTSBURGH — Schema


,dtype,nulls,null_%,expected,sample
column,,,,,
sample_id,int64,0,0.0%,✅,"[108, 7, 41]"
city,object,0,0.0%,✅,"['pittsburgh', 'pittsburgh', 'pittsburgh']"
instruction,object,0,0.0%,✅,['Meet me at the supermarket on Penn Avenue. It is the b...
oracle_label,object,0,0.0%,✅,"['Answerable', 'Answerable', 'Answerable']"
candidate_count,int64,0,0.0%,✅,"[14, 13, 259]"
start_node,object,0,0.0%,✅,"['#645296598', '#370352715', '#645305551']"
gold_goal_node,object,0,0.0%,✅,"['#6076721414', '#374188455', '#651244961']"
extracted_category,object,0,0.0%,✅,"['SHOP', 'SCHOOL', 'GARDEN']"
extracted_noun,object,309,30.2%,✅,"['supermarket', 'university', 'garden']"



  PHILADELPHIA — Schema


,dtype,nulls,null_%,expected,sample
column,,,,,
sample_id,int64,0,0.0%,✅,"[9126, 9127, 9128]"
city,object,0,0.0%,✅,"['philadelphia', 'philadelphia', 'philadelphia']"
instruction,object,0,0.0%,✅,"[""Meet to the west of you, at Ben & Jerry's ice cream on..."
oracle_label,object,0,0.0%,✅,"['Answerable', 'Contradictory', 'Answerable']"
candidate_count,int64,0,0.0%,✅,"[12, 0, 34]"
start_node,object,0,0.0%,✅,"['#6593007625', '#5723276180', '#1590298601']"
gold_goal_node,object,0,0.0%,✅,"['#2100740222', '#3275071095', '#4879082921']"
extracted_category,object,0,0.0%,✅,"['SHOP', 'UNKNOWN', 'MONUMENT']"
extracted_noun,object,254,19.9%,✅,"[""Ben & Jerry's ice cream"", 'historic memorial', 'cafe a..."


In [5]:
# Cell 5 — Null value heatmap across cities
null_data = {}
all_cols = set()
for city, df in dfs.items():
    all_cols.update(df.columns)

for col in sorted(all_cols):
    null_data[col] = {}
    for city, df in dfs.items():
        if col in df.columns:
            null_pct = df[col].isna().mean()
            null_data[col][city] = f"{null_pct:.1%}"
        else:
            null_data[col][city] = "❌ absent"

null_df = pd.DataFrame(null_data).T
null_df.index.name = "column"
print("=== NULL VALUES ACROSS CITIES ===")
print("(❌ = column absent, 0.0% = no nulls)")
display(null_df)

=== NULL VALUES ACROSS CITIES ===
(❌ = column absent, 0.0% = no nulls)


,manhattan,pittsburgh,philadelphia
column,,,
candidate_count,0.0%,0.0%,0.0%
city,0.0%,0.0%,0.0%
extracted_category,0.0%,0.0%,0.0%
extracted_direction,2.5%,3.8%,1.3%
extracted_noun,20.2%,30.2%,19.9%
gold_goal_node,0.0%,0.0%,0.0%
instruction,0.0%,0.0%,0.0%
oracle_label,0.0%,0.0%,0.0%
sample_id,0.0%,0.0%,0.0%


In [6]:
# Cell 6 — Key assumption validation
print("=== KEY ASSUMPTION VALIDATION ===\n")

for city, df in dfs.items():
    print(f"--- {city.upper()} ---")
    issues = []
    checks = []

    # 1. start_node present and non-null
    if 'start_node' in df.columns:
        null_ct = df['start_node'].isna().sum()
        checks.append(("start_node non-null",
                        "✅" if null_ct == 0 else f"⚠️  {null_ct} nulls"))
    else:
        issues.append("❌ start_node column MISSING")

    # 2. gold_goal_node present and non-null
    if 'gold_goal_node' in df.columns:
        null_ct = df['gold_goal_node'].isna().sum()
        checks.append(("gold_goal_node non-null",
                        "✅" if null_ct == 0 else f"⚠️  {null_ct} nulls"))
    else:
        issues.append("❌ gold_goal_node column MISSING")

    # 3. gold_goal_lat/lon present (enrichment step)
    for coord_col in ['gold_goal_lat', 'gold_goal_lon']:
        if coord_col in df.columns:
            null_ct = df[coord_col].isna().sum()
            checks.append((coord_col,
                            "✅" if null_ct == 0 else f"⚠️  {null_ct} nulls"))
        else:
            checks.append((coord_col, "❌ MISSING — run enrich_silver.py"))

    # 4. oracle_label only contains expected values
    if 'oracle_label' in df.columns:
        unexpected = set(df['oracle_label'].unique()) - {
            'Answerable', 'Contradictory', 'Ambiguous'}
        checks.append(("oracle_label values valid",
                        "✅" if not unexpected else f"⚠️  unexpected: {unexpected}"))

    # 5. extracted_category no unexpected UNKNOWN on Answerable rows
    if 'extracted_category' in df.columns and 'oracle_label' in df.columns:
        ans_df = df[df['oracle_label'] == 'Answerable']
        unknown_ct = (ans_df['extracted_category'] == 'UNKNOWN').sum()
        pct = unknown_ct / len(ans_df) if len(ans_df) > 0 else 0
        checks.append(("Answerable with UNKNOWN category",
                        f"✅ {unknown_ct} ({pct:.1%})" if pct < 0.05
                        else f"⚠️  {unknown_ct} ({pct:.1%}) — investigate"))

    # 6. candidate_count > 0 for all Answerable
    if 'candidate_count' in df.columns and 'oracle_label' in df.columns:
        bad = df[(df['oracle_label'] == 'Answerable') &
                 (df['candidate_count'] == 0)]
        checks.append(("Answerable has candidates > 0",
                        "✅" if len(bad) == 0
                        else f"❌ {len(bad)} Answerable rows have 0 candidates"))

    # 7. target_node present for Answerable rows
    if 'target_node' in df.columns and 'oracle_label' in df.columns:
        ans_missing = df[(df['oracle_label'] == 'Answerable') &
                         df['target_node'].isna()]
        checks.append(("Answerable has target_node",
                        "✅" if len(ans_missing) == 0
                        else f"❌ {len(ans_missing)} Answerable rows missing target_node"))

    # 8. No duplicate sample_ids
    if 'sample_id' in df.columns:
        dupes = df['sample_id'].duplicated().sum()
        checks.append(("No duplicate sample_ids",
                        "✅" if dupes == 0 else f"⚠️  {dupes} duplicates"))

    for check_name, result in checks:
        print(f"  {result:<45} {check_name}")
    for issue in issues:
        print(f"  {issue}")
    print()

=== KEY ASSUMPTION VALIDATION ===

--- MANHATTAN ---
  ✅                                             start_node non-null
  ✅                                             gold_goal_node non-null
  ❌ MISSING — run enrich_silver.py              gold_goal_lat
  ❌ MISSING — run enrich_silver.py              gold_goal_lon
  ✅                                             oracle_label values valid
  ⚠️  765 (14.0%) — investigate                 Answerable with UNKNOWN category
  ✅                                             Answerable has candidates > 0
  ✅                                             Answerable has target_node
  ⚠️  6489 duplicates                           No duplicate sample_ids

--- PITTSBURGH ---
  ✅                                             start_node non-null
  ✅                                             gold_goal_node non-null
  ❌ MISSING — run enrich_silver.py              gold_goal_lat
  ❌ MISSING — run enrich_silver.py              gold_goal_lon
  ✅                

In [7]:
# Cell 7 — Sample rows from each city (Answerable only)
for city, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"  {city.upper()} — Sample Answerable Rows")
    print(f"{'='*60}")

    cols_to_show = [c for c in [
        'sample_id', 'instruction', 'oracle_label',
        'extracted_category', 'extracted_noun', 'extracted_direction',
        'start_node', 'gold_goal_node', 'candidate_count'
    ] if c in df.columns]

    sample = df[df['oracle_label'] == 'Answerable'][cols_to_show].head(3)
    display(sample)


  MANHATTAN — Sample Answerable Rows


,sample_id,instruction,oracle_label,extracted_category,extracted_noun,extracted_direction,start_node,gold_goal_node,candidate_count
0,316,Can you meet me at the garden on Liberty Street. It's lo...,Answerable,GARDEN,garden,N,#5515156304,#697619118,122
1,140,Head northeast to meet me at the cafe on East 49th Stree...,Answerable,CAFE,cafe,NE,#2709347118,#2711466517,256
2,457,Meet me at the restaurant. Go northwest until you reach ...,Answerable,RESTAURANT,restaurant,NW,#8958523249,#3863997257,659



  PITTSBURGH — Sample Answerable Rows


,sample_id,instruction,oracle_label,extracted_category,extracted_noun,extracted_direction,start_node,gold_goal_node,candidate_count
0,108,Meet me at the supermarket on Penn Avenue. It is the bui...,Answerable,SHOP,supermarket,None,#645296598,#6076721414,14
1,7,After you get your haircut come meet me at the universit...,Answerable,SCHOOL,university,E,#370352715,#374188455,13
2,41,Get on Liberty Avenue past basketball pitch located on y...,Answerable,GARDEN,garden,NE,#645305551,#651244961,259



  PHILADELPHIA — Sample Answerable Rows


,sample_id,instruction,oracle_label,extracted_category,extracted_noun,extracted_direction,start_node,gold_goal_node,candidate_count
0,9126,"Meet to the west of you, at Ben & Jerry's ice cream on S...",Answerable,SHOP,Ben & Jerry's ice cream,W,#6593007625,#2100740222,12
2,9128,Meet me at the historic memorial on the south side of Ar...,Answerable,MONUMENT,historic memorial,S,#1590298601,#4879082921,34
3,9129,Go south and a bit east. You'll find me at the cafe acro...,Answerable,CAFE,cafe across,S,#2575197073,#3799448712,23


In [8]:
# Cell 8 — Cross-city schema diff
print("=== CROSS-CITY SCHEMA COMPARISON ===\n")

city_cols = {city: set(df.columns) for city, df in dfs.items()}
all_cols = set.union(*city_cols.values())

diff_rows = []
for col in sorted(all_cols):
    presence = {city: "✅" if col in city_cols[city] else "❌" for city in CITIES}
    consistent = len(set(presence.values())) == 1
    presence['consistent'] = "✅" if consistent else "⚠️  differs"
    presence['column'] = col
    diff_rows.append(presence)

diff_df = pd.DataFrame(diff_rows).set_index('column')
# Highlight inconsistent rows
inconsistent = diff_df[diff_df['consistent'] == "⚠️  differs"]
if len(inconsistent) > 0:
    print(f"⚠️  {len(inconsistent)} columns differ across cities:")
    display(inconsistent)
else:
    print("✅ All cities have identical schemas")

print(f"\nFull schema matrix ({len(diff_df)} columns):")
display(diff_df)

=== CROSS-CITY SCHEMA COMPARISON ===

✅ All cities have identical schemas

Full schema matrix (11 columns):


,manhattan,pittsburgh,philadelphia,consistent
column,,,,
candidate_count,✅,✅,✅,✅
city,✅,✅,✅,✅
extracted_category,✅,✅,✅,✅
extracted_direction,✅,✅,✅,✅
extracted_noun,✅,✅,✅,✅
gold_goal_node,✅,✅,✅,✅
instruction,✅,✅,✅,✅
oracle_label,✅,✅,✅,✅
sample_id,✅,✅,✅,✅


In [9]:
# Cell 9 — Final summary table
print("=== FINAL SUMMARY ===\n")

final_rows = []
for city, df in dfs.items():
    ans = (df['oracle_label'] == 'Answerable').sum() if 'oracle_label' in df.columns else 0
    row = {
        'city':              city,
        'total_rows':        len(df),
        'columns':           len(df.columns),
        'Answerable':        ans,
        'Answerable_%':      f"{ans/len(df):.1%}",
        'has_start_node':    "✅" if 'start_node' in df.columns else "❌",
        'has_goal_node':     "✅" if 'gold_goal_node' in df.columns else "❌",
        'has_coordinates':   "✅" if 'gold_goal_lat' in df.columns else "❌ run enrich",
        'has_target_node':   "✅" if 'target_node' in df.columns else "❌",
        'null_start_node':   f"{df['start_node'].isna().sum()}" if 'start_node' in df.columns else "N/A",
    }
    final_rows.append(row)

final_df = pd.DataFrame(final_rows).set_index('city')
display(final_df)

print("\n📋 Next steps based on validation:")
for city, df in dfs.items():
    if 'gold_goal_lat' not in df.columns:
        print(f"  ⚠️  {city}: run enrich_silver.py to add gold_goal_lat/lon")
    if 'start_node' in df.columns and df['start_node'].isna().sum() > 0:
        print(f"  ⚠️  {city}: {df['start_node'].isna().sum()} null start_nodes")

=== FINAL SUMMARY ===



,total_rows,columns,Answerable,Answerable_%,has_start_node,has_goal_node,has_coordinates,has_target_node,null_start_node
city,,,,,,,,,
manhattan,7000,11,5468,78.1%,✅,✅,❌ run enrich,✅,0
pittsburgh,1023,11,705,68.9%,✅,✅,❌ run enrich,✅,0
philadelphia,1278,11,990,77.5%,✅,✅,❌ run enrich,✅,0



📋 Next steps based on validation:
  ⚠️  manhattan: run enrich_silver.py to add gold_goal_lat/lon
  ⚠️  pittsburgh: run enrich_silver.py to add gold_goal_lat/lon
  ⚠️  philadelphia: run enrich_silver.py to add gold_goal_lat/lon


Auditing duplicate `sample_id` in Manhattan and Pittsburgh ⚠️

In [10]:
for city, df in dfs.items():
    if df['sample_id'].duplicated().sum() > 0:
        print(f"\n{city.upper()} — sample_id duplicate analysis:")
        dup_counts = df['sample_id'].value_counts()
        print(f"  Unique sample_ids: {df['sample_id'].nunique()}")
        print(f"  Total rows: {len(df)}")
        print(f"  Max occurrences of single id: {dup_counts.max()}")
        print(f"  IDs appearing >1 time: {(dup_counts > 1).sum()}")
        # Show example duplicates
        example_id = dup_counts.index[0]
        print(f"  Example duplicate id={example_id}:")
        display(df[df['sample_id'] == example_id][
            ['sample_id', 'instruction', 'oracle_label']].head(3))


MANHATTAN — sample_id duplicate analysis:
  Unique sample_ids: 511
  Total rows: 7000
  Max occurrences of single id: 33
  IDs appearing >1 time: 492
  Example duplicate id=32:


,sample_id,instruction,oracle_label
29,32,Meet me at the drinking water. Head south on 11th Avenue...,Answerable
171,32,Meet me at Artizia the women's clothing store right next...,Contradictory
595,32,"If you move west, you will see me at a pizza cafe by the...",Answerable



PITTSBURGH — sample_id duplicate analysis:
  Unique sample_ids: 462
  Total rows: 1023
  Max occurrences of single id: 8
  IDs appearing >1 time: 296
  Example duplicate id=312:


,sample_id,instruction,oracle_label
4,312,Meet me at the boutique shop on East Carson Street. It i...,Answerable
194,312,Cross the bridge over the river to the south and meet me...,Answerable
228,312,Meet me at the garden northeast of your location. Head n...,Contradictory


In [13]:
# Quick check — are keys actually unique in raw JSON?
import pandas as pd
for city in ['manhattan', 'pittsburgh', 'philadelphia']:
    try:
        df = pd.read_json(f'../data/{city}/{city}.json', lines=True)
    except:
        df = pd.read_json(f'../data/{city}/{city}.json')
    print(f"{city}: {len(df)} rows, {df['key'].nunique()} unique keys, "
          f"{df['rvs_sample_number'].nunique()} unique sample_numbers")

manhattan: 7000 rows, 7000 unique keys, 511 unique sample_numbers
pittsburgh: 1023 rows, 1023 unique keys, 462 unique sample_numbers


KeyError: 'rvs_sample_number'

In [14]:
# Check what fields each city actually has
import pandas as pd

for city in ['manhattan', 'pittsburgh', 'philadelphia']:
    try:
        df = pd.read_json(f'../data/{city}/{city}.json', lines=True)
    except:
        df = pd.read_json(f'../data/{city}/{city}.json')
    
    has_key = 'key' in df.columns
    has_sample_num = 'rvs_sample_number' in df.columns
    key_unique = df['key'].nunique() == len(df) if has_key else 'N/A'
    
    print(f"{city}: {len(df)} rows | "
          f"has 'key': {has_key} (unique={key_unique}) | "
          f"has 'rvs_sample_number': {has_sample_num}")
    print(f"  columns: {list(df.columns)}")
    print()

manhattan: 7000 rows | has 'key': True (unique=True) | has 'rvs_sample_number': True
  columns: ['rvs_sample_number', 'content', 'rvs_path', 'rvs_goal_point', 'key', 'region', 'rvs_start_point', 'landmarks']

pittsburgh: 1023 rows | has 'key': True (unique=True) | has 'rvs_sample_number': True
  columns: ['rvs_sample_number', 'content', 'rvs_path', 'rvs_goal_point', 'key', 'region', 'rvs_start_point', 'landmarks']

philadelphia: 1278 rows | has 'key': True (unique=True) | has 'rvs_sample_number': False
  columns: ['content', 'rvs_goal_point', 'key', 'region', 'rvs_start_point']

